# AuroraGate Expense Categorization
Improved pipeline: char n-grams · day-of-week features · merchant rules · LightGBM · 5-fold CV

In [3]:
# ── 0. Imports ────────────────────────────────────────────────────────────────
import re
import unicodedata
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb

from catboost import CatBoostClassifier
from xgboost import XGBClassifier

print('LightGBM:', lgb.__version__)

LightGBM: 4.7.0


In [4]:
# ── 1. Load data ──────────────────────────────────────────────────────────────
# BASE = '/kaggle/input/competitions/aurora-gate-expense-categorization-challenge'
BASE = './'
train = pd.read_csv(f'{BASE}/train.csv')
test  = pd.read_csv(f'{BASE}/test.csv')
print(f'Train: {train.shape}  Test: {test.shape}')
train.head()

Train: (8400, 6)  Test: (3600, 5)


,transaction_id,date,description,amount,day_of_week,category
0,1,2025-10-27,SPOTIFY USA NY,37.78,Monday,Entertainment
1,2,2025-02-13,WHOLE FOODS MKT #4854 CA,220.00,Thursday,Groceries
2,3,2025-11-20,CHIPOTLE 8319 WA,4.73,Thursday,Transportation
3,4,2025-07-26,SHELL OIL 6691 NY,46.57,Saturday,Transportation
4,5,2025-04-20,UBER *TRIP 4301,9.06,Sunday,Transportation


## Quick EDA

In [5]:
# ── 2. EDA ────────────────────────────────────────────────────────────────────
print(train['category'].value_counts())
print('\nAmount stats:')
print(train['amount'].describe())

category
Groceries            1191
Transportation       1138
Food & Dining        1096
Shopping             1082
Bills & Utilities     768
Entertainment         711
Health & Fitness      697
Miscellaneous         684
Subscriptions         599
Travel                434
Name: count, dtype: int64

Amount stats:
count    8400.000000
mean       53.349787
std        80.136906
min         3.000000
25%        12.960000
50%        27.350000
75%        59.977500
max       900.000000
Name: amount, dtype: float64


## Text Cleaning
Unicode normalisation + regex noise removal + whitespace collapsing.

In [6]:
# ── 3. Improved clean_text ────────────────────────────────────────────────────
def clean_text(s: str) -> str:
    """Normalise a transaction description string."""
    if not isinstance(s, str):
        return ''
    s = unicodedata.normalize('NFKC', s)    # full-width -> ASCII
    s = s.lower()
    s = re.sub(r'[#*\-_/\\|@&%$]', ' ', s) # strip noise chars
    s = re.sub(r'\s+', ' ', s)              # collapse whitespace
    return s.strip()

train['desc_clean'] = train['description'].apply(clean_text)
test['desc_clean']  = test['description'].apply(clean_text)

train[['description', 'desc_clean']].head()

,description,desc_clean
0,SPOTIFY USA NY,spotify usa ny
1,WHOLE FOODS MKT #4854 CA,whole foods mkt 4854 ca
2,CHIPOTLE 8319 WA,chipotle 8319 wa
3,SHELL OIL 6691 NY,shell oil 6691 ny
4,UBER *TRIP 4301,uber trip 4301


## Merchant Rules
For ambiguous merchants (e.g. UBER EATS vs UBER), a rule-match index is added
as an extra numeric feature — more specific patterns are listed first.

In [7]:
# ── 4. Merchant rules ─────────────────────────────────────────────────────────
# Order matters: specific patterns before generic ones
MERCHANT_RULES = [
    # Food & Dining
    ('uber eats',      'Food & Dining'),
    ('doordash',       'Food & Dining'),
    ('grubhub',        'Food & Dining'),
    ('seamless',       'Food & Dining'),
    ('postmates',      'Food & Dining'),
    # Transportation (listed AFTER 'uber eats')
    ('lyft',           'Transportation'),
    ('uber',           'Transportation'),
    # Subscriptions
    ('netflix',        'Subscriptions'),
    ('spotify',        'Subscriptions'),
    ('hulu',           'Subscriptions'),
    ('amazon prime',   'Subscriptions'),
    ('apple.com bill', 'Subscriptions'),
    # Groceries
    ('whole foods',    'Groceries'),
    ('instacart',      'Groceries'),
    ('trader joe',     'Groceries'),
    # Health & Fitness
    ('planet fitness', 'Health & Fitness'),
    ('cvs pharmacy',   'Health & Fitness'),
    ('walgreens',      'Health & Fitness'),
    # Bills & Utilities
    ('at&t',           'Bills & Utilities'),
    ('verizon',        'Bills & Utilities'),
    ('comcast',        'Bills & Utilities'),
    ('con edison',     'Bills & Utilities'),
]

RULE_CATS = sorted(set(c for _, c in MERCHANT_RULES))
RULE_IDX  = {c: i + 1 for i, c in enumerate(RULE_CATS)}  # 0 = no match

def rule_feature(desc: str) -> int:
    """Return the index of the first matching merchant rule, or 0."""
    for kw, cat in MERCHANT_RULES:
        if kw in desc:
            return RULE_IDX[cat]
    return 0

for df in [train, test]:
    df['rule_feat'] = df['desc_clean'].apply(rule_feature)

matched = (train['rule_feat'] > 0).sum()
print(f'Rule-matched rows: {matched}/{len(train)} ({matched/len(train):.1%})')

Rule-matched rows: 1843/8400 (21.9%)


## Day-of-Week & Weekend Features
`day_of_week` already exists in the data. We derive numeric, binary, and
cyclical encodings so the model captures weekly spending patterns.

In [8]:
# ── 5. Day-of-week / weekend features ─────────────────────────────────────────
DOW_MAP = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
    'Friday': 4, 'Saturday': 5, 'Sunday': 6,
}

for df in [train, test]:
    df['dow_num']    = df['day_of_week'].map(DOW_MAP).fillna(-1).astype(int)
    df['is_weekend'] = (df['dow_num'] >= 5).astype(int)
    # Cyclical encoding so Mon and Sun are numerically close
    df['dow_sin']    = np.sin(2 * np.pi * df['dow_num'].clip(0) / 7)
    df['dow_cos']    = np.cos(2 * np.pi * df['dow_num'].clip(0) / 7)
    # Amount features
    df['log_amount'] = np.log1p(df['amount'])

train[['day_of_week', 'dow_num', 'is_weekend', 'dow_sin', 'dow_cos']].head()

,day_of_week,dow_num,is_weekend,dow_sin,dow_cos
0,Monday,0,0,0.000000,1.000000
1,Thursday,3,0,0.433884,-0.900969
2,Thursday,3,0,0.433884,-0.900969
3,Saturday,5,1,-0.974928,-0.222521
4,Sunday,6,1,-0.781831,0.623490


## Feature Engineering
- **Word n-grams (1-2)**: standard TF-IDF over space-tokenised words
- **Character n-grams (3-5)**: captures merchant-code structure (e.g. `SHL`, `AMZN`)
- **Numeric**: amount, log-amount, DOW encodings, merchant rule index

In [9]:
# ── 6. Build feature matrix ───────────────────────────────────────────────────

# Word n-grams (1-2)
word_vec = TfidfVectorizer(
    max_features=2000, ngram_range=(1, 2),
    min_df=2, sublinear_tf=True,
)
X_word_tr = word_vec.fit_transform(train['desc_clean'])
X_word_te = word_vec.transform(test['desc_clean'])

# Character n-grams (3-5) - captures merchant code structure
char_vec = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5),
    max_features=3000, min_df=2, sublinear_tf=True,
)
X_char_tr = char_vec.fit_transform(train['desc_clean'])
X_char_te = char_vec.transform(test['desc_clean'])

# Numeric features
NUM_COLS = ['amount', 'log_amount', 'dow_num', 'is_weekend', 'dow_sin', 'dow_cos', 'rule_feat']
X_num_tr = csr_matrix(train[NUM_COLS].values.astype(float))
X_num_te = csr_matrix(test[NUM_COLS].values.astype(float))

# Stack everything
X_train = hstack([X_word_tr, X_char_tr, X_num_tr])
X_test  = hstack([X_word_te, X_char_te, X_num_te])

# Encode labels
le = LabelEncoder()
y_train = le.fit_transform(train['category'])

print(f'Feature matrix : {X_train.shape}')
print(f'Classes ({len(le.classes_)})  : {list(le.classes_)}')

Feature matrix : (8400, 4892)
Classes (10)  : ['Bills & Utilities', 'Entertainment', 'Food & Dining', 'Groceries', 'Health & Fitness', 'Miscellaneous', 'Shopping', 'Subscriptions', 'Transportation', 'Travel']


## 20-Fold Stratified Cross-Validation with LightGBM
Replaces the single train/validation split for a more reliable accuracy estimate.

In [ ]:
# ── 7. Cross-validate ─────────────────────────────────────────────────────────
lgbm = lgb.LGBMClassifier(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=63,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    min_child_samples=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(X_train.shape[0])

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train[train_idx], y_train[train_idx]
    X_va, y_va = X_train[val_idx], y_train[val_idx]
    
    cb = CatBoostClassifier(
        iterations=1500,
        learning_rate=0.05,
        depth=6,
        loss_function='MultiClass',
        eval_metric='TotalF1:average=Macro',
        auto_class_weights='Balanced',
        early_stopping_rounds=50,   # 50回連続でスコアが改善しなければ停止
        random_seed=42 + fold,
        thread_count=-1,
        verbose=0,                  # CV中のログ出力を抑える
    )
    
    cb.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        use_best_model=True
    )
    print(f"Fold {fold+1} Best Iteration: {cb.get_best_iteration()} | Best Score: {cb.get_best_score()['validation']['TotalF1:average=Macro']:.4f}")

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train[train_idx], y_train[train_idx]
    X_va, y_va = X_train[val_idx], y_train[val_idx]
    
    # サンプル重みの計算
    sample_weights_tr = compute_sample_weight('balanced', y_tr)
    
    xg = XGBClassifier(
        n_estimators=1500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        objective='multi:softprob',
        eval_metric='mlogloss',
        early_stopping_rounds=50,   # 50回連続で改善しなければ自動停止
        random_state=42 + fold,
        n_jobs=-1,
    )
    
    xg.fit(
        X_tr, y_tr,
        sample_weight=sample_weights_tr,
        eval_set=[(X_va, y_va)],
        verbose=False,
    )
    
    print(f"Fold {fold+1} Best Iteration: {xg.best_iteration} | Best Score: {xg.best_score:.4f}")

skf = StratifiedKFold(n_splits=20, shuffle=True, random_state=42)
cv_scores = cross_val_score(lgbm, X_train, y_train, cv=skf, scoring='f1_macro')

print(f'20-Fold CV Accuracy : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
print(f'Per-fold           : {cv_scores.round(4)}')


## Train Final Model on Full Training Set

In [ ]:
# ── 8. Train on full data & predict ───────────────────────────────────────────
lgbm.fit(X_train, y_train)
preds = le.inverse_transform(lgbm.predict(X_test))

print('Prediction distribution:')
print(pd.Series(preds).value_counts())

## Create Submission

In [ ]:
# ── 9. Submission ─────────────────────────────────────────────────────────────
submission = pd.DataFrame({
    'transaction_id': test['transaction_id'],
    'category':       preds,
})
submission.to_csv('submission.csv', index=False)
submission.head(10)